# Fleurs data prep
Download and preprocess Fleurs data, upload to GCS bucket for use by Label Studio
1) load_dataset
2) Apply mfa_g2p mapping
3) strip suprasegmental features
4) Add gcs_url column
5) Copy audio files to your local folder
6) Drop audio column
7) Save CSVs
8) Upload audio to GCS Bucket

# Setup

In [1]:
from transformers import Wav2Vec2Processor

/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from huggingface_hub import login, logout
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from datasets import load_dataset
import numpy as np

In [3]:
import os
from dotenv import load_dotenv, dotenv_values 
# loading variables from .env file
load_dotenv() 
HF_login = os.getenv("HF_KEY")

login(HF_login)

# fetch data and inspect

In [4]:
import torch
import soundfile as sf
import torchaudio

In [5]:
from datasets import load_dataset
train = load_dataset("google/fleurs", "ga_ie", split="train", trust_remote_code=True)
val = load_dataset("google/fleurs", "ga_ie", split="validation", trust_remote_code=True)
test = load_dataset("google/fleurs", "ga_ie", split="test", trust_remote_code=True)

In [6]:
train[0]

{'id': 571,
 'num_samples': 172800,
 'path': '/home/peter/.cache/huggingface/datasets/downloads/extracted/cf2f05dbc6d7c83a2ca5f3625f601bff35293b374f8cbe9c8d98e3829f3a201b/10009174761044778838.wav',
 'audio': {'path': 'train/10009174761044778838.wav',
  'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00030464,
         -0.00026166, -0.00036532], shape=(172800,)),
  'sampling_rate': 16000},
 'transcription': 'nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe tugtar daonra monómorfach orthu',
 'raw_transcription': 'Nuair a bhíonn tréith feinitíopach ar leith i bpáirt ag gach duine i ndaonra áirithe, tugtar daonra monómorfach orthu.',
 'gender': 0,
 'lang_id': 27,
 'language': 'Irish',
 'lang_group_id': 0}

In [7]:
audio_input = train[0]["audio"]

In [8]:
audio_input

{'path': 'train/10009174761044778838.wav',
 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00030464,
        -0.00026166, -0.00036532], shape=(172800,)),
 'sampling_rate': 16000}

In [9]:
from IPython.display import Audio

# Play some sample. (audio doesn't work for me period in vscode)
Audio(data=audio_input['array'], rate=16000, autoplay=True)

# load g2p dict

In [10]:
import pandas as pd

In [11]:
os.listdir("../../../data")

['idlak', 'teanglann', 'phoneme_vocab.json', 'fleurs', 'common_voice']

In [12]:
g2p_path="../../../models/mfa/abair_g2p/ulster.tsv"
g2p_file = pd.read_csv(g2p_path,sep="\t", names=["word","phonemes"])
# turn df into dict for simple lookup
g2p_dict = g2p_file.set_index("word")["phonemes"].to_dict()

# Add phoneme transcriptions

## mapping function


In [13]:
# helper for mapping function
import tempfile
import os

def mfa_fallback(words, model_path="../../../models/mfa/abair_g2p/g2p-ulster.zip"):
    """
    Make MFA g2p to generate phonemes for OOV words
    Args:
        words (_type_): _description_
        model_path (str, optional): _description_. Defaults to "../../../models/mfa/abair_g2p/g2p-ulster.zip".

    Returns:
        _type_: _description_
    """
    # create temporary files for input/output
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
        input_file = f.name
        for word in words:
            f.write(word + "\n")
    
    fd, output_file = tempfile.mkstemp(suffix='.txt')
    os.close(fd)
    
    try:
        # Run MFA to get phonemes. mfa g2p [OPTIONS] INPUT_PATH G2P_MODEL_PATH OUTPUT_PATH
        !mfa g2p {input_file} {model_path} {output_file}   
    
        # Load results into fallback dict
        ipa_fallbacks = {}
        with open(output_file, encoding="utf-8") as f:
            output_lines = [line.strip() for line in f if line.strip()]
        
        for orig_word, line in zip(words,output_lines):
            parts = line.strip().split("\t")
            if len(parts) == 2:
                ipa_fallbacks[orig_word] = parts[1]
        
        return ipa_fallbacks
    
    finally:
        # Clean up temp files
        if os.path.exists(input_file):
            os.remove(input_file)
        if os.path.exists(output_file[1]):
            os.remove(output_file[1])

In [14]:
def gather_oov(data, oov_set):
    # just get oovs from all datasets.
    for example in data:
        word_list = [x.strip(" .,!?:;") for x in example["transcription"].split()]
        oov_set.update(w for w in word_list if w not in g2p_dict and w.lower() not in g2p_dict)
    return oov_set

In [15]:
# collect all oovs to not call mfa every time you run into one (slow as hell)
oovs = set()
oovs = gather_oov(train, oovs)
oovs = gather_oov(val, oovs)
oovs = gather_oov(test, oovs)

ipa_fallbacks = mfa_fallback(list(oovs))

 INFO     Generating pronunciations...                                          
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,636/2,636  [ 0:00:05 < 0:00:00 , 478 it/s ]499 it/s ]
 INFO     Done! Everything took 9.630 seconds                                   


In [16]:
# The function needs to accept and output a dict
def mfa_g2p(example):
    word_list = [x.strip(" .,!?:;") for x in example["transcription"].split()]


    # lookup and fallback logic
    phoneme_seq = []
    for word in word_list:
        # if in dict...
        if word in g2p_dict: 
            phoneme_seq.append(g2p_dict[word])
        elif word.lower() in g2p_dict:
            phoneme_seq.append(g2p_dict[word.lower()])
        elif word in ipa_fallbacks:
            phoneme_seq.append(ipa_fallbacks[word])
        else:
            phoneme_seq.append("[UNK]")            
            
    example["phonetic"] = " ".join(phoneme_seq)              
    return example

In [17]:
mfa_g2p({"transcription":"ottowa"})

{'transcription': 'ottowa', 'phonetic': '[UNK]'}

train.map(mfa_g2p, load_from_cache_file=False) to prevent saving tons of datasets during developments. .map creates cached versions that can stack across runs.


In [18]:
#train = train.map(mfa_g2p)
#val = val.map(mfa_g2p)
test = test.map(mfa_g2p)

Parameter 'fn_kwargs'={} of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.
Map: 100%|██████████| 842/842 [00:01<00:00, 437.87 examples/s]


In [19]:
test[0]

{'id': 1777,
 'num_samples': 381120,
 'path': '/home/peter/.cache/huggingface/datasets/downloads/extracted/bf9a03c8938ccf8c73329df6bf727510d839d97feb21b27824c4e534175f8d9b/10045899353945045473.wav',
 'audio': {'path': 'test/10045899353945045473.wav',
  'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          1.25408173e-04, -9.06586647e-05, -2.86519527e-04], shape=(381120,)),
  'sampling_rate': 16000},
 'transcription': 'is é ottawa príomhchathair ghalánta dhátheangach cheanada agus gheofar ann dánlanna agus músaeim ealaíne ann a thaispeánann ceanada san am a chuaigh thart agus san am i láthair',
 'raw_transcription': 'Is é Ottawa príomhchathair ghalánta, dhátheangach Cheanada agus gheofar ann dánlanna agus músaeim ealaíne ann a thaispeánann Ceanada san am a chuaigh thart agus san am i láthair.',
 'gender': 0,
 'lang_id': 27,
 'language': 'Irish',
 'lang_group_id': 0,
 'phonetic': 'ˈ i sˠ ˈ eː pˠ ɾˠ vʲ a n̠ʲ z n̪ˠ ʌ ˈ pʲ ɾʲ iː vˠ ˈ x a h ə ɾʲ ɣ a l̻ˠ aː n̻ˠ t̪ˠ

## strip suprasegmental features

In [20]:
primary_stress = "ˈ"
secondary_stress = "ˌ"
suprasegmentals = {primary_stress, secondary_stress}
condition = lambda x: x in suprasegmentals  
def strip_suprasegmentals(example):
    phon_string = example['phonetic']
    phone_list = phon_string.split()
    stripped_list = list(filter(lambda x: not condition(x), phone_list))
    stripped_string = " ".join(stripped_list)
    example['phonetic'] = stripped_string
    return example

In [21]:
phonetics = mfa_g2p({"transcription":"dia duit"})
print(phonetics)
print(strip_suprasegmentals(phonetics))

{'transcription': 'dia duit', 'phonetic': 'ˈ dʲ ia ˈ d̪ˠ i tʲ'}
{'transcription': 'dia duit', 'phonetic': 'dʲ ia d̪ˠ i tʲ'}


In [22]:
#train = train.map(strip_suprasegmentals)
#val = val.map(strip_suprasegmentals)
test = test.map(strip_suprasegmentals)

Map: 100%|██████████| 842/842 [00:02<00:00, 379.38 examples/s] 


# Save to disk

In [23]:
os.listdir("../../../data/fleurs")

['train',
 'test',
 'transcription',
 'test.csv',
 'val',
 'audio',
 'train.csv',
 'val.csv',
 'fleurs_phonemes.csv']

In [24]:
# first add gcs bucket path to dataset. maybe unnecessary?
bucket = os.getenv("GCS_EVAL_BUCKET")

def add_gcs_path(example):
    filename = os.path.basename(example["path"])
    example["gcs_url"] = f"gs://{bucket}/fleurs/audio/{filename}"
    return example

In [25]:
#train = train.map(add_gcs_path)
#val = val.map(add_gcs_path)
test = test.map(add_gcs_path)

Map: 100%|██████████| 842/842 [00:01<00:00, 430.98 examples/s]


To reload, use...

from datasets import load_from_disk

reloaded_encoded_dataset = load_from_disk("path/of/my/dataset/directory")

In [26]:
# save audio locally for upload with gsutil
import soundfile as sf
import os

audio_out = "../../../data/fleurs/audio"
os.makedirs(audio_out, exist_ok=True)

split_dir = os.path.join(audio_out, "test")
os.makedirs(split_dir, exist_ok=True)
for example in test:
    src = example["path"]  # path to .wav in HF cache
    dst = os.path.join(split_dir, os.path.basename(src))
    sf.write(dst, example["audio"]["array"], example["audio"]["sampling_rate"])


In [27]:
# audio column is massive and doesn't save properly as csv anyways, so just axe it.
cols_to_drop = ["audio"]
fleurs_path = "../../../data/fleurs"

#train.remove_columns(cols_to_drop).to_csv(fleurs_path + "/train.csv")
#val.remove_columns(cols_to_drop).to_csv(fleurs_path + "/val.csv")
test.remove_columns(cols_to_drop).to_csv(fleurs_path + "/test.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 23.80ba/s]


755016

# upload to gcs

In [ ]:
#!gsutil -m cp -r ../../../data/fleurs/audio gs://{bucket}/audio



Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://../../../data/fleurs/audio/test/17312170155992694161.wav [Content-Type=audio/x-wav]...
Copying file://../../../data/fleurs/audio/test/9862503142410451972.wav [Content-Type=audio/x-wav]...
Copying file://../../../data/fleurs/audio/test/11045120557578453196.wav [Content-Type=audio/x-wav]...
Copying file://../../../data/fleurs/audio/test/5427575755498535681.wav [Content-Type=audio/x-wav]...
Copying file://../../../data/fleurs/audio/test/8150659789222277052.wav [Content-Type=audio/x-wav]...
Copying file://../../../data/fleurs/audio/test/11364508537898331159.wav [Content-Type=audio/x-wav]...
Copying fi